# EDA Pipeline Runner
Этот ноутбук воспроизводит пайплайн EDA для задачи детекции фразы "не слышу".

## 0. Клонирование репозитория и установка зависимостей

In [ ]:
%%bash
set -e
if [ ! -d Vseros_A ]; then
  git clone <REPO_URL> Vseros_A
fi
cd Vseros_A
python3 -m pip install --upgrade pip
pip install -r EDA/requirements.txt

## 1. Переход в репозиторий

In [ ]:
%cd Vseros_A

In [ ]:
import json
import os
import re
import subprocess
from pathlib import Path

import requests
from tqdm.auto import tqdm

MAIL_RU_TRAIN = "https://cloud.mail.ru/public/Gsyr/8VxmbhAaZ/train_data.tar"
MAIL_RU_TEST = "https://cloud.mail.ru/public/Gsyr/8VxmbhAaZ/test_data.tar"

def get_direct_file_link(mailru_file_url: str) -> str:
    resp = requests.get(mailru_file_url)
    resp.raise_for_status()
    match = re.search(r'dispatcher.*?weblink_get.*?url":"(.*?)"', resp.text)
    if not match:
        raise RuntimeError("Не удалось найти CDN ссылку")
    base_url = match.group(1)
    parts = mailru_file_url.strip("/").split("/")[-3:]
    return f"{base_url}/{parts[0]}/{parts[1]}/{parts[2]}"

def download_from_mailru(file_url: str, local_path: Path):
    if local_path.exists():
        print(f"Файл {local_path} уже скачан")
        return local_path
    direct = get_direct_file_link(file_url)
    print(f"Скачиваем {file_url} → {local_path}")
    with requests.get(direct, stream=True) as r:
        r.raise_for_status()
        total_size = int(r.headers.get("content-length", 0))
        block = 8192
        with open(local_path, "wb") as f, tqdm(total=total_size, unit="B", unit_scale=True) as bar:
            for chunk in r.iter_content(block):
                f.write(chunk)
                bar.update(len(chunk))
    return local_path

def extract_tar(archive: Path, target: Path):
    target.mkdir(parents=True, exist_ok=True)
    subprocess.run(["tar", "xf", str(archive), "-C", str(target)], check=True)

def sync_dir(src: Path, dst: Path):
    dst.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["rsync", "-a", str(src), str(dst)], check=True)

raw_root = Path('/content/data/raw')
raw_root.mkdir(parents=True, exist_ok=True)
train_tar = download_from_mailru(MAIL_RU_TRAIN, raw_root / 'train_data.tar')
test_tar = download_from_mailru(MAIL_RU_TEST, raw_root / 'test_data.tar')

print('Распаковка архивов...')
extract_tar(train_tar, raw_root)
extract_tar(test_tar, raw_root)

workspace = Path('/content') / 'eda_workspace'
(train_dst := workspace / 'data/raw').mkdir(parents=True, exist_ok=True)
(meta_dst := workspace / 'data/meta').mkdir(parents=True, exist_ok=True)

sync_dir(raw_root / 'train_opus', workspace / 'data/raw')
sync_dir(raw_root / 'test_opus', workspace / 'data/raw')
word_bounds = raw_root / 'train_opus' / 'word_bounds.json'
if word_bounds.exists():
    subprocess.run(["cp", str(word_bounds), str(meta_dst / 'word_bounds.json')], check=True)

print('Готово: данные загружены и скопированы в eda_workspace')


## 3. Запуск EDA пайплайна

In [ ]:
%%bash
set -e
cd Vseros_A
python -m EDA.run prepare --config EDA/configs/default.yaml --force
python -m EDA.run download --config EDA/configs/default.yaml --force
python -m EDA.run inventory --config EDA/configs/default.yaml --force
python -m EDA.run labels --config EDA/configs/default.yaml --force
python -m EDA.run convert --config EDA/configs/default.yaml --force
python -m EDA.run vad --config EDA/configs/default.yaml --force
python -m EDA.run negatives --config EDA/configs/default.yaml --force
python -m EDA.run duplicates --config EDA/configs/default.yaml --force
python -m EDA.run cv --config EDA/configs/default.yaml --force
python -m EDA.run windowing --config EDA/configs/default.yaml --force
python -m EDA.run slices --config EDA/configs/default.yaml --force
python -m EDA.run report --config EDA/configs/default.yaml --force

## 4. Обзор артефактов

In [ ]:
from pathlib import Path
root = Path('/content/eda_workspace')
for path in sorted(root.glob('**/*')):
    if path.is_file():
        print(path)
